# reentry_label_conditioned_bc_diagnostic_v1

Purpose: test whether the under-participation windows identified by the aggregate OOS diagnostic can be converted into no-leak walk-forward behavior-cloning labels, before opening another PPO training branch.

Governance:
- Baseline: `current_runtime_paper` daily replay from `runtime/production/card/paper_model_oos_risk_check_v1/paper_model_oos_risk_check_v1_soxl_daily.csv`.
- Direct comparison: same replay with predicted behavior-clone labels adjusting daily `position`.
- Role: `diagnose`; no actor checkpoint, no promotion.
- Changed layer: `model_selection` / label design. Reusable label construction lives in `src.soxl_quant.app.reentry_labels`; classifier/grid logic stays in this notebook.
- Target windows: no-leak walk-forward windows `wf_2022`, `wf_2023_2024`, and bad clusters C002/C003. `wf_2020` is not used for no-leak classifier scoring because the replay label history starts in 2020.
- Acceptance: any candidate must improve mean window return by at least 30pp and keep every scored key-window MaxDD <= 15%.

External references used as design constraints:
- Qlib paper and GitHub for reproducible research harness discipline: https://arxiv.org/abs/2009.11189 and https://github.com/microsoft/qlib
- Probability of Backtest Overfitting and Deflated Sharpe for keeping future-aware oracle labels separate from promotion evidence: https://scholarworks.wmich.edu/math_pubs/42/ and https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551
- Combinatorial Purged Cross-Validation / financial ML validation discipline: https://www.quantresearch.org/Innovations.htm
- Behavior-cloning trading-agent direction: https://arxiv.org/abs/2110.01325
- Momentum crash/rebound mechanism behind violent post-stress rallies: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2371227

Twitter/X note: no tweet is used as evidence or gate input here. Social posts can suggest ideas, but this notebook uses citable papers and repo data for decisions.


In [1]:
from __future__ import annotations

import json
from dataclasses import asdict
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.soxl_quant.app.budget_counterfactual import compute_metrics
from src.soxl_quant.app.oos_diagnostics import load_qlib_feature_frame
from src.soxl_quant.app.reentry_labels import ReentryOracleLabelConfig, build_reentry_oracle_label_frame

EXPERIMENT = "reentry_label_conditioned_bc_diagnostic_v1"
OUTPUT_DIR = Path("runtime/model/card") / EXPERIMENT
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DAILY_CSV = Path("runtime/production/card/paper_model_oos_risk_check_v1/paper_model_oos_risk_check_v1_soxl_daily.csv")
AGG_DIR = Path("runtime/model_selection/card/oos_poor_window_factor_diff_aggregate_v1/aggregate")

WINDOWS = {
    "wf_2022": ("2022-01-01", "2022-12-31"),
    "wf_2023_2024": ("2023-01-01", "2024-12-31"),
    "bad_C002_2025_rebound": ("2025-04-04", "2025-07-22"),
    "bad_C003_2026_reentry": ("2026-02-04", "2026-05-21"),
}

FEATURE_COLS = [
    "vol_of_vol_21",
    "tracking_error_vol_21",
    "decay_63",
    "realized_vol_63",
    "tracking_error_21",
    "realized_vol_21",
    "mom_21",
    "mom_10",
    "relative_strength_21",
    "vix_percentile_252",
    "anchor_gap_180",
    "below_anchor_ratio_63",
    "regime_risk",
    "drawdown_63",
    "downside_semivol_21",
    "vix_zscore",
    "trend_gap_21",
    "trend_gap_63",
    "realized_vol_10",
]

daily = pd.read_csv(DAILY_CSV, parse_dates=["datetime"]).sort_values("datetime").reset_index(drop=True)
price = daily.set_index("datetime")["price"].astype(float)
features = load_qlib_feature_frame(str(daily["datetime"].min().date()), str(daily["datetime"].max().date()))
features = features.reset_index().rename(columns={"index": "datetime"})
features["datetime"] = pd.to_datetime(features["datetime"])
poor_clusters = pd.read_csv(AGG_DIR / "poor_window_clusters.csv")
feature_clusters = pd.read_csv(AGG_DIR / "feature_delta_clusters.csv")

print({
    "daily_rows": len(daily),
    "daily_start": str(daily["datetime"].min().date()),
    "daily_end": str(daily["datetime"].max().date()),
    "features_rows": len(features),
    "feature_cols": len(FEATURE_COLS),
})
poor_clusters.head()

[25787:MainThread](2026-05-25 12:35:15,679) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[25787:MainThread](2026-05-25 12:35:16,349) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[25787:MainThread](2026-05-25 12:35:16,349) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[25787:MainThread](2026-05-25 12:35:18,867) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Loading data Done


[25787:MainThread](2026-05-25 12:35:18,871) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[25787:MainThread](2026-05-25 12:35:18,874) INFO - qlib.timer - [log.py:127] - Time cost: 0.009s | Init data Done


{'daily_rows': 1605, 'daily_start': '2020-01-02', 'daily_end': '2026-05-21', 'features_rows': 1605, 'feature_cols': 19}


,cluster_id,cluster_start,cluster_end,profile_count,window_count,profiles,mean_excess_return,worst_excess_return,mean_max_dd,max_max_dd,mean_avg_pos,mean_bad_score,max_bad_score,dominant_failure_type,primary_optimization_path
0,C002,2025-04-04,2025-07-22,5,30,"balanced_rule_residual_v19_reg,balanced_rule_r...",-160.502243,-199.393437,5.924443,16.324924,0.519568,166.426685,203.420490,underparticipation,reentry_participation_or_behavior_clone
1,C003,2026-02-04,2026-05-21,4,18,"balanced_rule_residual_v22,current_runtime_pap...",-167.300896,-198.889223,10.842927,24.118142,0.089418,178.143823,205.310696,underparticipation,reentry_participation_or_behavior_clone
2,C001,2020-03-18,2020-06-19,4,12,"balanced_rule_residual_v19_reg,balanced_rule_r...",-158.096497,-174.787005,8.683677,13.885509,0.514949,166.780175,182.311302,underparticipation,reentry_participation_or_behavior_clone


In [2]:
def score_path(frame: pd.DataFrame, start: str, end: str, position_col: str = "position") -> dict:
    sub = frame[(frame["datetime"] >= start) & (frame["datetime"] <= end)].copy()
    return compute_metrics(sub[["datetime", position_col]].rename(columns={position_col: "position"}), price.loc[start:end])

def base_window_metrics() -> pd.DataFrame:
    rows = []
    for name, (start, end) in WINDOWS.items():
        rows.append({"window": name, "variant": "baseline", **score_path(daily, start, end, "position")})
    return pd.DataFrame(rows)

def flatten_metrics(prefix: str, metrics: dict) -> dict:
    return {
        f"{prefix}_return": metrics["return"],
        f"{prefix}_relative_return": metrics["relative_return"],
        f"{prefix}_max_dd": metrics["max_dd"],
        f"{prefix}_avg_pos": metrics["avg_pos"],
        f"{prefix}_trades": metrics["trades"],
    }

base_metrics = base_window_metrics()
base_metrics[["window", "return", "relative_return", "max_dd", "avg_pos", "trades"]]

,window,return,relative_return,max_dd,avg_pos,trades
0,wf_2022,-21.567182,65.020890,31.096849,0.248277,26
1,wf_2023_2024,209.141657,17.368148,30.668184,0.497231,70
2,bad_C002_2025_rebound,76.882481,-124.606649,16.324924,0.572851,9
3,bad_C003_2026_reentry,17.799626,-217.582972,6.421473,0.161722,2


## Oracle Upper Bounds

The first check is intentionally future-aware and **not promotable**. It asks whether a target label family could in principle close the gap. If an oracle label cannot move the metrics, training a classifier on that label is pointless.

Two label families are checked:
- `reentry_only`: lift low/medium base positions before future positive, low-forward-drawdown rebounds.
- `dual_reentry_derisk`: the same re-entry label plus a future-drawdown de-risk label. This is a diagnostic upper bound for a two-action behavior policy.

In [3]:
def oracle_position_frame(cfg: ReentryOracleLabelConfig, *, derisk_dd: float | None = None, derisk_target: float = 0.0) -> pd.DataFrame:
    labels = build_reentry_oracle_label_frame(daily, cfg)
    labels["oracle_class"] = 0
    labels.loc[labels["oracle_reentry"], "oracle_class"] = 1
    labels["oracle_position"] = labels["expert_position"]
    if derisk_dd is not None:
        derisk = (
            labels["has_forward_window"]
            & (labels["base_position"] >= 0.05)
            & ((labels["forward_drawdown"] >= float(derisk_dd)) | (labels["forward_return"] <= -0.05))
        )
        labels.loc[derisk, "oracle_class"] = -1
        labels.loc[derisk, "oracle_position"] = np.minimum(labels.loc[derisk, "base_position"], float(derisk_target))
    return labels

oracle_rows = []
ORACLE_CONFIGS = [
    ("reentry_only_21d", ReentryOracleLabelConfig(21, 0.55, 0.75, 0.40, 0.18), None, 0.0),
    ("reentry_only_10d", ReentryOracleLabelConfig(10, 0.55, 0.75, 0.40, 0.10), None, 0.0),
    ("dual_21d_dd10", ReentryOracleLabelConfig(21, 0.65, 0.75, 0.40, 0.18), 0.10, 0.0),
    ("dual_42d_dd10", ReentryOracleLabelConfig(42, 0.65, 0.75, 0.40, 0.18), 0.10, 0.0),
]
for label, cfg, derisk_dd, derisk_target in ORACLE_CONFIGS:
    oracle = oracle_position_frame(cfg, derisk_dd=derisk_dd, derisk_target=derisk_target)
    full_metrics = compute_metrics(oracle[["datetime", "oracle_position"]].rename(columns={"oracle_position": "position"}), price)
    row = {
        "label_family": label,
        "derisk_dd": derisk_dd,
        "derisk_target": derisk_target,
        "reentry_labels": int((oracle["oracle_class"] == 1).sum()),
        "derisk_labels": int((oracle["oracle_class"] == -1).sum()),
        **flatten_metrics("oracle_full", full_metrics),
    }
    for window, (start, end) in WINDOWS.items():
        oracle_window_metrics = score_path(oracle, start, end, "oracle_position")
        row[f"{window}_return"] = oracle_window_metrics["return"]
        row[f"{window}_max_dd"] = oracle_window_metrics["max_dd"]
    oracle_rows.append(row)

oracle_results = pd.DataFrame(oracle_rows)
oracle_results.to_csv(OUTPUT_DIR / "oracle_upper_bounds.csv", index=False)
oracle_results[["label_family", "reentry_labels", "derisk_labels", "oracle_full_return", "oracle_full_max_dd", "oracle_full_avg_pos", "wf_2022_max_dd", "wf_2023_2024_max_dd"]]

,label_family,reentry_labels,derisk_labels,oracle_full_return,oracle_full_max_dd,oracle_full_avg_pos,wf_2022_max_dd,wf_2023_2024_max_dd
0,reentry_only_21d,130,0,1.062949e+04,40.889077,0.496104,26.600820,30.668184
1,reentry_only_10d,42,0,8.312180e+03,40.092027,0.476302,24.815391,30.668184
2,dual_21d_dd10,133,895,2.763869e+06,12.904572,0.251546,7.099409,7.899390
3,dual_42d_dd10,185,1032,4.953723e+05,8.210775,0.214210,7.099409,7.899390


## No-Leak Walk-Forward Classifier

Classifier setup is deliberately simple and regularized: `StandardScaler + LogisticRegression(C=0.05 or 0.2, class_weight=balanced)`. Each scored window trains only on replay-label rows before the window start.

This is not meant to be the final model architecture; it is a leakage check and label learnability screen before investing in PPO or a sequence network.

In [4]:
def labeled_feature_frame(cfg: ReentryOracleLabelConfig, *, derisk_dd: float | None = None, derisk_target: float = 0.0) -> pd.DataFrame:
    labels = oracle_position_frame(cfg, derisk_dd=derisk_dd, derisk_target=derisk_target)
    frame = labels.merge(features[["datetime", *FEATURE_COLS]], on="datetime", how="left")
    return frame.dropna(subset=FEATURE_COLS).reset_index(drop=True)

def apply_predictions(test: pd.DataFrame, pred: np.ndarray, cfg: ReentryOracleLabelConfig, derisk_target: float | None) -> pd.DataFrame:
    path = test[["datetime", "base_position"]].rename(columns={"base_position": "position"}).copy()
    reentry_mask = pred == 1
    path.loc[reentry_mask, "position"] = np.maximum(path.loc[reentry_mask, "position"], float(cfg.target_position))
    if derisk_target is not None:
        derisk_mask = pred == -1
        path.loc[derisk_mask, "position"] = np.minimum(path.loc[derisk_mask, "position"], float(derisk_target))
    return path

def score_classifier_family(
    *,
    family: str,
    cfg: ReentryOracleLabelConfig,
    thresholds: list[float],
    derisk_dd: float | None = None,
    derisk_target: float | None = None,
    c_value: float = 0.2,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    frame = labeled_feature_frame(cfg, derisk_dd=derisk_dd, derisk_target=derisk_target or 0.0)
    window_paths: dict[str, pd.DataFrame] = {}
    rows = []
    for threshold in thresholds:
        row = {
            "family": family,
            "threshold": threshold,
            "c_value": c_value,
            "derisk_dd": derisk_dd,
            "derisk_target": derisk_target,
            **{f"cfg_{k}": v for k, v in asdict(cfg).items()},
            "total_reentry_predictions": 0,
            "total_derisk_predictions": 0,
        }
        deltas = []
        max_dds = []
        for window, (start, end) in WINDOWS.items():
            train = frame[(frame["datetime"] < start) & frame["has_forward_window"]].copy()
            test = frame[(frame["datetime"] >= start) & (frame["datetime"] <= end)].copy()
            if len(train) < 100 or train["oracle_class"].nunique() < 2 or len(test) < 2:
                row[f"{window}_status"] = "insufficient_train_labels"
                continue
            clf = make_pipeline(
                StandardScaler(),
                LogisticRegression(C=float(c_value), class_weight="balanced", max_iter=1000, random_state=7),
            )
            clf.fit(train[FEATURE_COLS], train["oracle_class"].astype(int))
            proba = clf.predict_proba(test[FEATURE_COLS])
            classes = list(clf.named_steps["logisticregression"].classes_)
            pred = np.zeros(len(test), dtype=int)
            if 1 in classes:
                p_reentry = proba[:, classes.index(1)]
                pred[(p_reentry >= threshold) & (test["base_position"] <= float(cfg.max_base_position))] = 1
            if derisk_target is not None and -1 in classes:
                p_derisk = proba[:, classes.index(-1)]
                pred[(p_derisk >= threshold) & (test["base_position"] >= 0.05)] = -1

            path = apply_predictions(test, pred, cfg, derisk_target)
            metrics = score_path(path, start, end, "position")
            base_frame = test[["datetime", "base_position"]].rename(columns={"base_position": "position"})
            base = score_path(base_frame, start, end, "position")
            row[f"{window}_return"] = metrics["return"]
            row[f"{window}_delta_return"] = metrics["return"] - base["return"]
            row[f"{window}_max_dd"] = metrics["max_dd"]
            row[f"{window}_delta_max_dd"] = metrics["max_dd"] - base["max_dd"]
            row[f"{window}_avg_pos"] = metrics["avg_pos"]
            row[f"{window}_reentry_predictions"] = int((pred == 1).sum())
            row[f"{window}_derisk_predictions"] = int((pred == -1).sum())
            row["total_reentry_predictions"] += int((pred == 1).sum())
            row["total_derisk_predictions"] += int((pred == -1).sum())
            deltas.append(metrics["return"] - base["return"])
            max_dds.append(metrics["max_dd"])
            window_paths[f"{family}_thr{threshold}_{window}"] = path
        row["mean_delta_return"] = float(np.mean(deltas)) if deltas else float("nan")
        row["max_key_window_dd"] = float(np.max(max_dds)) if max_dds else float("nan")
        row["return_gate"] = bool(row["mean_delta_return"] >= 30.0) if deltas else False
        row["maxdd_gate"] = bool(row["max_key_window_dd"] <= 15.0) if max_dds else False
        row["diagnostic_score"] = row["mean_delta_return"] - 3.0 * max(row["max_key_window_dd"] - 15.0, 0.0)
        rows.append(row)
    return pd.DataFrame(rows), window_paths

reentry_rows = []
dual_rows = []
all_paths = {}
for cfg, c_value in [
    (ReentryOracleLabelConfig(21, 0.55, 0.75, 0.40, 0.18), 0.2),
    (ReentryOracleLabelConfig(21, 0.55, 0.65, 0.40, 0.18), 0.2),
    (ReentryOracleLabelConfig(10, 0.55, 0.75, 0.40, 0.10), 0.2),
]:
    rows, paths = score_classifier_family(
        family="reentry_only", cfg=cfg, thresholds=[0.50, 0.65, 0.75], c_value=c_value
    )
    reentry_rows.append(rows)
    all_paths.update(paths)

for cfg, derisk_dd, c_value in [
    (ReentryOracleLabelConfig(21, 0.65, 0.75, 0.40, 0.18), 0.10, 0.05),
    (ReentryOracleLabelConfig(21, 0.65, 0.75, 0.40, 0.18), 0.15, 0.05),
    (ReentryOracleLabelConfig(10, 0.65, 0.75, 0.40, 0.10), 0.10, 0.05),
]:
    rows, paths = score_classifier_family(
        family="dual_reentry_derisk",
        cfg=cfg,
        thresholds=[0.45, 0.55, 0.65, 0.75],
        derisk_dd=derisk_dd,
        derisk_target=0.0,
        c_value=c_value,
    )
    dual_rows.append(rows)
    all_paths.update(paths)

reentry_results = pd.concat(reentry_rows, ignore_index=True).sort_values("diagnostic_score", ascending=False)
dual_results = pd.concat(dual_rows, ignore_index=True).sort_values("diagnostic_score", ascending=False)
reentry_results.to_csv(OUTPUT_DIR / "reentry_classifier_results.csv", index=False)
dual_results.to_csv(OUTPUT_DIR / "dual_classifier_results.csv", index=False)

print("reentry rows", len(reentry_results), "dual rows", len(dual_results))
reentry_results.head(8)[["family", "threshold", "cfg_horizon_days", "cfg_max_base_position", "cfg_target_position", "mean_delta_return", "max_key_window_dd", "return_gate", "maxdd_gate", "total_reentry_predictions", "diagnostic_score"]]

reentry rows 9 dual rows 12


,family,threshold,cfg_horizon_days,cfg_max_base_position,cfg_target_position,mean_delta_return,max_key_window_dd,return_gate,maxdd_gate,total_reentry_predictions,diagnostic_score
7,reentry_only,0.65,10,0.55,0.75,30.634898,45.491658,True,False,190,-60.840076
5,reentry_only,0.75,21,0.55,0.65,7.049649,41.352352,False,False,81,-72.007406
2,reentry_only,0.75,21,0.55,0.75,8.480803,45.346378,False,False,81,-82.558332
8,reentry_only,0.75,10,0.55,0.75,6.987898,44.981094,False,False,156,-82.955384
4,reentry_only,0.65,21,0.55,0.65,0.763971,43.277127,False,False,115,-84.067409
3,reentry_only,0.50,21,0.55,0.65,4.425872,44.732454,False,False,179,-84.771491
0,reentry_only,0.50,21,0.55,0.75,5.914584,48.846712,False,False,179,-95.625551
1,reentry_only,0.65,21,0.55,0.75,-0.037718,47.753962,False,False,115,-98.299603


In [5]:
dual_results.head(10)[[
    "family", "threshold", "cfg_horizon_days", "derisk_dd", "mean_delta_return", "max_key_window_dd",
    "return_gate", "maxdd_gate", "total_reentry_predictions", "total_derisk_predictions",
    "wf_2022_delta_return", "wf_2022_max_dd", "wf_2023_2024_delta_return", "wf_2023_2024_max_dd",
    "bad_C002_2025_rebound_delta_return", "bad_C002_2025_rebound_max_dd",
    "bad_C003_2026_reentry_delta_return", "bad_C003_2026_reentry_max_dd",
    "diagnostic_score",
]]

,family,threshold,cfg_horizon_days,derisk_dd,mean_delta_return,max_key_window_dd,return_gate,maxdd_gate,total_reentry_predictions,total_derisk_predictions,wf_2022_delta_return,wf_2022_max_dd,wf_2023_2024_delta_return,wf_2023_2024_max_dd,bad_C002_2025_rebound_delta_return,bad_C002_2025_rebound_max_dd,bad_C003_2026_reentry_delta_return,bad_C003_2026_reentry_max_dd,diagnostic_score
7,dual_reentry_derisk,0.75,21,0.15,14.011824,30.668184,False,False,18,7,28.154532,28.358209,-1.212551,30.668184,29.105315,12.478720,0.000000,6.421473,-32.992729
3,dual_reentry_derisk,0.75,21,0.10,11.819167,30.668184,False,False,20,6,17.628267,28.358209,2.025143,30.668184,27.623258,12.478720,0.000000,6.421473,-35.185386
2,dual_reentry_derisk,0.65,21,0.10,15.556637,34.868512,False,False,58,17,9.900887,34.868512,49.079220,30.668184,3.246442,12.478720,0.000000,6.421473,-44.048899
6,dual_reentry_derisk,0.65,21,0.15,9.835051,37.052767,False,False,54,19,6.938520,37.052767,42.589922,30.668184,-4.488925,12.478720,-5.699312,6.421473,-56.323251
1,dual_reentry_derisk,0.55,21,0.10,4.783275,37.827772,False,False,93,43,-0.799939,37.827772,30.957810,30.668184,-5.673891,12.478720,-5.350881,6.421473,-63.700043
9,dual_reentry_derisk,0.55,10,0.10,13.470860,40.942077,False,False,109,20,11.917894,40.942077,15.574376,30.668184,32.015091,12.478720,-5.623923,7.283898,-64.355370
5,dual_reentry_derisk,0.55,21,0.15,10.485438,40.804757,False,False,94,54,-1.269232,40.804757,66.590880,30.668184,-5.673891,12.478720,-17.706006,0.000000,-66.928834
8,dual_reentry_derisk,0.45,10,0.10,34.153434,57.286072,True,False,157,115,-15.959793,57.286072,163.457680,32.186489,-6.544643,12.478720,-4.339506,0.000000,-92.704780
4,dual_reentry_derisk,0.45,21,0.15,13.102900,50.345335,False,False,146,116,-9.363185,50.345335,123.209953,30.002416,-43.635544,12.478720,-17.799626,0.000000,-92.933106
11,dual_reentry_derisk,0.75,10,0.10,0.024120,46.060002,False,False,42,3,-4.711399,46.060002,0.000000,30.668184,4.807879,16.324924,0.000000,6.421473,-93.155885


In [6]:
combined = pd.concat([reentry_results, dual_results], ignore_index=True).sort_values("diagnostic_score", ascending=False)
selected = combined.iloc[0].to_dict()
selected_key = None
for key, path_df in all_paths.items():
    if key.startswith(selected["family"] + f"_thr{selected['threshold']}"):
        selected_key = key
        break

comparison_rows = []
for window, (start, end) in WINDOWS.items():
    base = score_path(daily, start, end, "position")
    comparison_rows.append({"window": window, "variant": "baseline", **base})

# Rebuild selected paths per window from the saved all_paths map.
for key, path_df in all_paths.items():
    if not key.startswith(selected["family"] + f"_thr{selected['threshold']}"):
        continue
    window = key.split("_", 3)[-1]
    if window not in WINDOWS:
        # Family names contain underscores; recover by suffix matching below.
        for known in WINDOWS:
            if key.endswith("_" + known):
                window = known
                break
    start, end = WINDOWS[window]
    comparison_rows.append({"window": window, "variant": "selected_classifier", **score_path(path_df, start, end, "position")})

comparison = pd.DataFrame(comparison_rows).sort_values(["window", "variant"])
comparison.to_csv(OUTPUT_DIR / "selected_comparison.csv", index=False)

selection = {
    "experiment": EXPERIMENT,
    "decision": "diagnostic_not_promoted",
    "reason": "future-aware dual oracle can satisfy drawdown target, but no-leak logistic BC labels do not keep key-window MaxDD <= 15%; reentry-only labels also cannot solve drawdown",
    "baseline_daily_csv": str(DAILY_CSV),
    "oracle_upper_bounds_csv": str(OUTPUT_DIR / "oracle_upper_bounds.csv"),
    "reentry_classifier_results_csv": str(OUTPUT_DIR / "reentry_classifier_results.csv"),
    "dual_classifier_results_csv": str(OUTPUT_DIR / "dual_classifier_results.csv"),
    "selected_comparison_csv": str(OUTPUT_DIR / "selected_comparison.csv"),
    "selected_classifier": {k: (None if pd.isna(v) else v) for k, v in selected.items() if isinstance(v, (int, float, str, bool)) or pd.isna(v)},
    "next_path": "do not open broad PPO retrain; first design a stronger no-leak sequence/state model or a better de-risk label, because simple linear BC cannot learn the oracle safely",
}
(OUTPUT_DIR / "selection.json").write_text(json.dumps(selection, indent=2, ensure_ascii=True), encoding="utf-8")
comparison[["window", "variant", "return", "relative_return", "max_dd", "avg_pos", "trades"]]

/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_25787/100526986.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([reentry_results, dual_results], ignore_index=True).sort_values("diagnostic_score", ascending=False)


,window,variant,return,relative_return,max_dd,avg_pos,trades
2,bad_C002_2025_rebound,baseline,76.882481,-124.606649,16.324924,0.572851,9
6,bad_C002_2025_rebound,selected_classifier,81.690360,-119.798771,16.324924,0.562716,12
3,bad_C003_2026_reentry,baseline,17.799626,-217.582972,6.421473,0.161722,2
7,bad_C003_2026_reentry,selected_classifier,17.799626,-217.582972,6.421473,0.161722,2
0,wf_2022,baseline,-21.567182,65.020890,31.096849,0.248277,26
4,wf_2022,selected_classifier,-26.278581,60.309491,46.060002,0.332416,43
1,wf_2023_2024,baseline,209.141657,17.368148,30.668184,0.497231,70
5,wf_2023_2024,selected_classifier,209.141657,17.368148,30.668184,0.497231,70


In [7]:
best_oracle = oracle_results.sort_values("oracle_full_max_dd").iloc[0].to_dict()
best_reentry = reentry_results.iloc[0].to_dict()
best_dual = dual_results.iloc[0].to_dict()
conclusion = f"""## Runtime Conclusion

Decision: `diagnostic_not_promoted`.

Oracle readout: future-aware dual labels can reduce full replay MaxDD to {best_oracle['oracle_full_max_dd']:.2f}% while producing very large return, but this is future leakage and only proves that the label family has theoretical capacity. Re-entry-only oracle/classifier variants do not address the global drawdown problem.

Best no-leak re-entry classifier: mean window delta {best_reentry['mean_delta_return']:.2f}pp, max key-window DD {best_reentry['max_key_window_dd']:.2f}%, return_gate={best_reentry['return_gate']}, maxdd_gate={best_reentry['maxdd_gate']}.

Best no-leak dual classifier: mean window delta {best_dual['mean_delta_return']:.2f}pp, max key-window DD {best_dual['max_key_window_dd']:.2f}%, return_gate={best_dual['return_gate']}, maxdd_gate={best_dual['maxdd_gate']}. The dual classifier either misses de-risk timing or over-de-risks rebound windows, so it is not safe to promote.

Keep/drop: keep `src.soxl_quant.app.reentry_labels` as reusable no-promotion label construction. Drop this simple logistic BC path as a promotion candidate. Next valid experiment is a stronger sequence/state learner or a cleaner two-stage label design: first classify de-risk states with purged/rolling validation, then classify re-entry states only after drawdown risk clears.

Artifacts: `oracle_upper_bounds.csv`, `reentry_classifier_results.csv`, `dual_classifier_results.csv`, `selected_comparison.csv`, and `selection.json`.
"""
display(Markdown(conclusion))
print(json.dumps({
    "best_oracle_family": best_oracle["label_family"],
    "best_oracle_full_return": best_oracle["oracle_full_return"],
    "best_oracle_full_max_dd": best_oracle["oracle_full_max_dd"],
    "best_reentry_mean_delta": best_reentry["mean_delta_return"],
    "best_reentry_max_dd": best_reentry["max_key_window_dd"],
    "best_dual_mean_delta": best_dual["mean_delta_return"],
    "best_dual_max_dd": best_dual["max_key_window_dd"],
}, indent=2))

## Runtime Conclusion

Decision: `diagnostic_not_promoted`.

Oracle readout: future-aware dual labels can reduce full replay MaxDD to 8.21% while producing very large return, but this is future leakage and only proves that the label family has theoretical capacity. Re-entry-only oracle/classifier variants do not address the global drawdown problem.

Best no-leak re-entry classifier: mean window delta 30.63pp, max key-window DD 45.49%, return_gate=True, maxdd_gate=False.

Best no-leak dual classifier: mean window delta 14.01pp, max key-window DD 30.67%, return_gate=False, maxdd_gate=False. The dual classifier either misses de-risk timing or over-de-risks rebound windows, so it is not safe to promote.

Keep/drop: keep `src.soxl_quant.app.reentry_labels` as reusable no-promotion label construction. Drop this simple logistic BC path as a promotion candidate. Next valid experiment is a stronger sequence/state learner or a cleaner two-stage label design: first classify de-risk states with purged/rolling validation, then classify re-entry states only after drawdown risk clears.

Artifacts: `oracle_upper_bounds.csv`, `reentry_classifier_results.csv`, `dual_classifier_results.csv`, `selected_comparison.csv`, and `selection.json`.


{
  "best_oracle_family": "dual_42d_dd10",
  "best_oracle_full_return": 495372.34564836754,
  "best_oracle_full_max_dd": 8.210775219447312,
  "best_reentry_mean_delta": 30.63489829403942,
  "best_reentry_max_dd": 45.491658200819614,
  "best_dual_mean_delta": 14.011823822455554,
  "best_dual_max_dd": 30.668184146246208
}
